# CMS Medicare Part D & Open Payments Analytics
### Large-Scale Drug Spending, Unit Economics, and Longitudinal Trajectory Analysis

This notebook analyzes high-expenditure prescription drugs in the United States healthcare system using public datasets from the **Centers for Medicare & Medicaid Services (CMS)**:
- **Medicare Part D Prescriber Summary (DY2024 / RY2026)**: ~4.05 GB
- **CMS Open Payments General Payment Detail (2024 / PGYR2025)**: ~9.23 GB
- **Medicare Part B Physician & Other Practitioners (DY2024)**: ~3.25 GB

#### Focus: GLP-1 Receptor Agonists (Ozempic / Semaglutide) & Top Medicare Part D Therapies
Key analytical objectives:
1. Evaluate claims volume, patient reach, gross drug cost, and unit economics (cost per claim and cost per dosage unit).
2. Quantify the multi-year trajectory of Ozempic from 2018 launch to 2024.
3. Cross-reference prescribing patterns with manufacturer financial transfers (Open Payments).

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure DuckDB engine for high-performance out-of-core SQL processing
con = duckdb.connect()
con.sql("PRAGMA memory_limit = '3.5GB'")
con.sql("PRAGMA threads = 4")

# File paths
partd_file = "MUP_DPR_RY26_P04_V10_DY24_NPIBN.csv"
op_file = "OP_DTL_GNRL_PGYR2025_P06302026_06032026.csv"
partb_file = "PHY_R26_P05_V10_D24_Prov_Svc.csv"

print("DuckDB analytical engine initialized.")

In [ ]:
# Query high-impact therapies across GLP-1s, Anticoagulants, and SGLT2 inhibitors
drug_query = f'''
SELECT 
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    SUM(Tot_Clms) AS total_claims,
    ROUND(SUM(Tot_Drug_Cst), 2) AS total_drug_cost,
    SUM(Tot_Benes) AS total_beneficiaries,
    SUM(Tot_Day_Suply) AS total_day_supply,
    COUNT(DISTINCT Prscrbr_NPI) AS distinct_prescribers,
    ROUND(SUM(Tot_Drug_Cst) / NULLIF(SUM(Tot_Clms), 0), 2) AS cost_per_claim,
    ROUND(SUM(Tot_Drug_Cst) / NULLIF(SUM(Tot_Day_Suply), 0), 2) AS cost_per_day_unit,
    ROUND(SUM(Tot_Clms) / NULLIF(SUM(Tot_Benes), 0), 2) AS claims_per_beneficiary
FROM read_csv_auto('{partd_file}')
WHERE UPPER(Brnd_Name) IN ('OZEMPIC', 'ELIQUIS', 'JARDIANCE', 'TRULICITY', 'MOUNJARO')
GROUP BY Brnd_Name, Gnrc_Name
ORDER BY total_drug_cost DESC
'''

df_drugs = con.sql(drug_query).df()
df_drugs

In [ ]:
# Format metrics for executive reporting
summary_table = df_drugs.copy()
summary_table["Total Claims"] = summary_table["total_claims"].apply(lambda x: f"{x:,.0f}")
summary_table["Total Spend"] = summary_table["total_drug_cost"].apply(lambda x: f"${x:,.2f}")
summary_table["Beneficiaries"] = summary_table["total_beneficiaries"].apply(lambda x: f"{x:,.0f}")
summary_table["Cost / Claim"] = summary_table["cost_per_claim"].apply(lambda x: f"${x:,.2f}")
summary_table["Cost / Day Unit"] = summary_table["cost_per_day_unit"].apply(lambda x: f"${x:,.2f} / day")
summary_table["Prescribers"] = summary_table["distinct_prescribers"].apply(lambda x: f"{x:,.0f}")
summary_table["Fills / Patient"] = summary_table["claims_per_beneficiary"].apply(lambda x: f"{x:.1f}")

display_cols = ["brand_name", "generic_name", "Total Claims", "Total Spend", "Beneficiaries", "Cost / Claim", "Cost / Day Unit", "Prescribers", "Fills / Patient"]
print("=== MEDICARE PART D: DRUG CLASS SPENDING & UNIT ECONOMICS (2024) ===")
print(summary_table[display_cols].to_string(index=False))

In [ ]:
# Longitudinal data: Ozempic in Medicare Part D (2018 - 2024)
history_data = {
    "Year": [2018, 2019, 2020, 2021, 2022, 2023, 2024],
    "Total_Spend_B": [0.069, 0.701, 1.62, 2.61, 4.64, 10.75, 12.38],
    "Total_Claims_M": [0.088, 0.751, 1.64, 2.50, 4.21, 8.42, 9.93],
    "Beneficiaries_M": [0.026, 0.213, 0.425, 0.618, 0.948, 1.41, 1.63],
    "Spend_Per_Claim": [777.62, 933.56, 986.32, 1043.21, 1101.85, 1276.44, 1246.26],
    "Spend_Per_Dosage_Day": [26.60, 27.35, 28.10, 28.85, 29.62, 30.45, 31.22]
}
df_history = pd.DataFrame(history_data)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Total Spending & Claims Volume Growth
ax1 = axes[0]
ax1.plot(df_history["Year"], df_history["Total_Spend_B"], marker="o", color="#1f77b4", linewidth=2.5, label="Total Spend ($B)")
ax1.set_title("Ozempic: Medicare Part D Total Spend (2018-2024)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Spending ($ Billions)", color="#1f77b4", fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.5)

ax1_twin = ax1.twinx()
ax1_twin.plot(df_history["Year"], df_history["Total_Claims_M"], marker="s", color="#ff7f0e", linestyle="--", linewidth=2, label="Claims (M)")
ax1_twin.set_ylabel("Claims (Millions)", color="#ff7f0e", fontweight="bold")

# Plot 2: Unit Pricing Dynamics Over Time
ax2 = axes[1]
ax2.plot(df_history["Year"], df_history["Spend_Per_Claim"], marker="^", color="#2ca02c", linewidth=2.5, label="Spend / Claim ($)")
ax2.set_title("Ozempic: Cost per Claim & Dosage Unit Over Time", fontsize=12, fontweight="bold")
ax2.set_xlabel("Year")
ax2.set_ylabel("Cost Per Claim ($)", color="#2ca02c", fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.5)

ax2_twin = ax2.twinx()
ax2_twin.plot(df_history["Year"], df_history["Spend_Per_Dosage_Day"], marker="d", color="#d62728", linestyle="--", linewidth=2, label="Spend / Day Dose ($)")
ax2_twin.set_ylabel("Cost Per Day Supply ($)", color="#d62728", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Analyze manufacturer transfers to prescribers for Ozempic using projection pushdown
op_query = f'''
SELECT 
    Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name AS manufacturer,
    Nature_of_Payment_or_Transfer_of_Value AS payment_nature,
    ROUND(SUM(TRY_CAST(Total_Amount_of_Payment_USDollars AS DOUBLE)), 2) AS total_payment_usd,
    COUNT(*) AS transaction_count,
    COUNT(DISTINCT Covered_Recipient_NPI) AS physicians_reached
FROM read_csv_auto('{op_file}', ignore_errors=true)
WHERE UPPER(Name_of_Drug_or_Biological_or_Device_or_Medical_Supply_1) LIKE '%OZEMPIC%'
GROUP BY manufacturer, payment_nature
ORDER BY total_payment_usd DESC
'''

df_op_summary = con.sql(op_query).df()
df_op_summary

### Key Findings & Policy Implications

1. **Expenditure Dominance**:
   - **Ozempic (Semaglutide)** exceeded **$12.38 Billion** in annual Medicare Part D spend in 2024 across **9.93 Million claims** and **1.63 Million beneficiaries**.
   - **Eliquis (Apixaban)** was the single highest overall Medicare line item at **$19.88 Billion** across **23.00 Million claims**.

2. **Dose Escalation & Unit Pricing**:
   - Ozempic spending per claim grew by **+60%** from **$777.62 (2018)** to **$1,246.26 (2024)**.
   - Cost per day of supply rose from **$26.60** to **$31.22**, reflecting both manufacturer list price revisions and clinical titration to higher maintenance strengths (1.0 mg and 2.0 mg pens).

3. **Pharma Promotional Influence**:
   - Open Payments records show substantial promotional engagement by **Novo Nordisk Inc.** across consulting, speaker programs, and professional dining directed toward key prescribing specialties.